# AMPS II · Week 08 — Lab
## Reissner–Mindlin plates: shear locking, reduced integration, and the right reference

| | |
|---|---|
| **Lecture sections** | Part A (Reissner–Mindlin, the Kirchhoff limit) · Part B (locking and its cures) · Part C (references and verification) · Part D (LLM) |
| **Time** | ≈ 60 min in class (second half of the session), finish at home — no homework this week |
| **You will** | reproduce the source's locking table · see that reduced integration does **not** cure triangles · measure the thickness effect against exact Reissner–Mindlin solutions · measure convergence rates · check an LLM's circular plate |

**Reminder.** `%%fenicsx` cells are separate MPI programs; ordinary cells have NumPy and matplotlib only. The model module `plate.py` is written once with
`%%writefile` (§1). All plate cells run with `-np 2`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The model module  *(lecture §A1–A2, §B1–B3, §C1)*
`Plate(N, deg, family, shear, h, E, nu, q, bc, cell)`: the unit square with $N\times N$ quadrilaterals (`family` `"Q"` Lagrange or `"S"` serendipity) or triangles, the same degree
for $w$ and $\boldsymbol\theta$, the shear term with the full rule or the reduced one (`"sri"`: quadrature degree $2\,\mathrm{deg}-2$), boundary conditions `"clamped"`, `"ss_hard"`, `"ss_soft"`;
`.solve()`, `.center()` (point evaluation — serendipity DOFs are not all point values). `navier_ss(D, F)`: the exact centre deflection of the simply supported square
(Kirchhoff for $F=\infty$, the hard simply supported Reissner–Mindlin plate otherwise). Run the cell.

In [ ]:
%%writefile /tmp/fnx/plate.py
"""Reissner-Mindlin plate on the unit square, DOLFINx 0.11. Unknowns (w, theta), w = deflection, theta = rotation of the normal.
Bending  D [(1 - nu) chi + nu tr(chi) I] : chi(psi),  chi = sym grad theta,  D = E h^3 / (12 (1 - nu^2));
shear    F (grad w - theta) . (grad v - psi),        F = kappa G h, kappa = 5/6.
Thin limit h -> 0: grad w = theta (Kirchhoff). Equal-order elements LOCK unless the shear term is integrated with a reduced rule."""
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from dolfinx import fem, mesh, geometry
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

W_CLAMPED_K = 1.265319087e-3            # clamped square, Kirchhoff: w_centre = W_CLAMPED_K q a^4 / D

class Plate:
    """unit square, N x N cells; cell "quad" (family "Q" Lagrange or "S" serendipity) or "tri" (Lagrange); degree deg for w and theta;
    shear = "full" or "sri" (shear term with quadrature degree 2 deg - 2: one point for degree 1 on quadrilaterals);
    bc = "clamped" (w = theta = 0), "ss_hard" (w = 0 and the tangential rotation = 0), "ss_soft" (w = 0 only); uniform load q downward."""
    count = 0
    def __init__(self, N, deg=1, family="Q", shear="sri", h=1e-3, E=210e3, nu=0.3, q=1.0, bc="clamped", cell="quad"):
        ct = mesh.CellType.quadrilateral if cell == "quad" else mesh.CellType.triangle
        msh = self.msh = mesh.create_unit_square(comm, N, N, ct)
        c = msh.basix_cell(); fam = family if cell == "quad" else "Lagrange"
        V = self.V = fem.functionspace(msh, basix.ufl.mixed_element([basix.ufl.element(fam, c, deg), basix.ufl.element(fam, c, deg, shape=(2,))]))
        self.D = E * h**3 / (12 * (1 - nu**2)); self.F = 5 / 6 * E / (2 * (1 + nu)) * h; self.q = q
        D, F = self.D, self.F
        (w, th), (v, ps) = ufl.TrialFunctions(V), ufl.TestFunctions(V)
        chi = lambda t: ufl.sym(ufl.grad(t))
        M = lambda t: D * ((1 - nu) * chi(t) + nu * ufl.tr(chi(t)) * ufl.Identity(2))
        dx = ufl.Measure("dx", domain=msh)
        dxs = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": 2 * deg - 2}) if shear == "sri" else dx
        a = ufl.inner(M(th), chi(ps)) * dx + F * ufl.dot(ufl.grad(w) - th, ufl.grad(v) - ps) * dxs
        L = -q * v * dx
        side = lambda x: np.isclose(x[0], 0) | np.isclose(x[0], 1) | np.isclose(x[1], 0) | np.isclose(x[1], 1)
        bcs = [self._zero(V.sub(0), side)]
        if bc == "clamped":
            bcs.append(self._zero(V.sub(1), side))
        elif bc == "ss_hard":                                  # tangential rotation: theta_y on x = 0, 1; theta_x on y = 0, 1
            bcs.append(self._zero(V.sub(1).sub(1), lambda x: np.isclose(x[0], 0) | np.isclose(x[0], 1)))
            bcs.append(self._zero(V.sub(1).sub(0), lambda x: np.isclose(x[1], 0) | np.isclose(x[1], 1)))
        Plate.count += 1
        self.u = fem.Function(V)
        self.problem = LinearProblem(a, L, u=self.u, bcs=bcs, petsc_options_prefix=f"plate{Plate.count}_",
                                     petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"})
        self.ndofs = V.dofmap.index_map.size_global * V.dofmap.index_map_bs

    def _zero(self, Vs, where):
        Vc, _ = Vs.collapse()
        dofs = fem.locate_dofs_topological((Vs, Vc), 1, mesh.locate_entities_boundary(self.msh, 1, where))
        return fem.dirichletbc(fem.Function(Vc), dofs, Vs)

    def solve(self):
        self.problem.solve()
        return self.problem.solver.getConvergedReason()

    def at(self, x, y):
        """downward deflection -w at a point: point evaluation in the owning cell (any element, also serendipity, whose DOFs
        are not all point values), summed over the ranks"""
        p = np.array([[x, y, 0.0]])
        cells = geometry.compute_colliding_cells(self.msh, geometry.compute_collisions_points(geometry.bb_tree(self.msh, 2), p), p)
        val, cnt = 0.0, 0
        if len(cells.links(0)) and cells.links(0)[0] < self.msh.topology.index_map(2).size_local:
            val, cnt = -float(self.u.sub(0).eval(p, [cells.links(0)[0]])[0]), 1
        return comm.allreduce(val, op=MPI.SUM) / max(comm.allreduce(cnt, op=MPI.SUM), 1)

    def center(self):
        return self.at(0.5, 0.5)

def navier_ss(D, F=np.inf, q=1.0, terms=401):
    """simply supported unit square, uniform load q, centre deflection. F = inf: Kirchhoff (Navier double sine series).
    Finite F: the hard simply supported Reissner-Mindlin plate, exactly w_RM = w_K + M/F with the moment sum M = (Mxx + Myy)/(1 + nu)
    of the Kirchhoff plate (Marcus), M solving -Lap M = q, M = 0 on the boundary."""
    m = np.arange(1, terms + 1, 2)
    mm, nn = np.meshgrid(m, m, indexing="ij")
    qmn = 16 * q / (np.pi**2 * mm * nn); k2 = np.pi**2 * (mm**2 + nn**2)
    s = np.sin(mm * np.pi / 2) * np.sin(nn * np.pi / 2)
    wK = np.sum(qmn / (D * k2**2) * s); Ms = np.sum(qmn / k2 * s)
    return wK + (Ms / F if np.isfinite(F) else 0.0)

## 2 · Locking on quadrilaterals  *(lecture §B1–B2)*
The source's test: clamped unit square, $E=210\times10^3$, $\nu=0.3$, $h=10^{-3}$ (span/thickness 1000), uniform load. Centre deflection over the Kirchhoff value $1.265319\times10^{-3}\,qa^4/D$
(at this thickness the Reissner–Mindlin answer differs from it by $2\times10^{-5}$ — §4).

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from plate import *

E, nu, h = 210e3, 0.3, 1e-3
wK = W_CLAMPED_K / (E * h**3 / (12 * (1 - nu**2)))
for fam, deg in (("Q", 1), ("S", 2), ("Q", 2)):
    for shear in ("full", "sri"):
        row = []
        for N in (10, 50):
            p = Plate(N, deg, fam, shear, h=h); p.solve(); row.append(f"N = {N}: {p.center() / wK:.5f} ({p.ndofs} DOFs)")
        say(f"{fam}{deg} {shear:4s}: " + "   ".join(row))

> **✏️ Your turn (5 min).** Which elements lock, and how badly? Why does Q2 with the full rule do much better than Q1 with the full rule? Why is S2 (8 nodes) with the reduced
> rule worse than Q2 (9 nodes) with it on the coarse mesh?

> *your answer*

## 3 · Triangles: reduced integration is not a cure  *(lecture §B3)*
The same plate on $N\times N\times2$ triangles, Lagrange P1, P2, P3 for $w$ and $\boldsymbol\theta$, full and reduced shear rule.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from plate import *

E, nu, h = 210e3, 0.3, 1e-3
wK = W_CLAMPED_K / (E * h**3 / (12 * (1 - nu**2)))
for deg in (1, 2, 3):
    for shear in ("full", "sri"):
        row = []
        for N in (8, 16, 32):
            p = Plate(N, deg, shear=shear, h=h, cell="tri"); p.solve(); row.append(f"{p.center() / wK:.5f}")
        say(f"P{deg} triangles, {shear:4s}: N = 8, 16, 32: " + ", ".join(row))

> **✏️ Your turn (5 min).** Count constraints: in the thin limit $\nabla w=\boldsymbol\theta$ must hold at every shear quadrature point. Why does one point per cell save the bilinear quadrilateral but
> not the linear triangle? What would you use on a triangle mesh?

> *your answer*

## 4 · Thickness: which reference?  *(lecture §A2, §C1)*
**Simply supported (hard):** the exact Reissner–Mindlin deflection is $w_{RM}=w_K+M/F$ (Kirchhoff plus the moment sum over the shear stiffness) — `navier_ss(D, F)`. **Clamped:**
no closed form; the Q2-SRI result on a fine mesh. The source compared everything with Kirchhoff, also at $h=0.1$ (span/thickness 10), and reported "SRI 1.184, naive 0.993".

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from plate import *

E, nu = 210e3, 0.3
for h in (0.1, 0.01, 0.001):
    D = E * h**3 / (12 * (1 - nu**2)); F = 5 / 6 * E / (2 * (1 + nu)) * h
    wK, wRM = navier_ss(D), navier_ss(D, F)
    row = []
    for deg, shear in ((1, "sri"), (1, "full"), (2, "sri")):
        p = Plate(20, deg, shear=shear, h=h, bc="ss_hard"); p.solve(); row.append(f"Q{deg}-{shear} {p.center() / wRM:.6f}")
    say(f"h = {h}: simply supported  w_RM / w_K = {wRM / wK:.5f};  N = 20, over w_RM: " + ", ".join(row))
    ref = Plate(64, 2, h=h); ref.solve(); wc = ref.center(); wKc = W_CLAMPED_K / D
    q1 = Plate(10, 1, h=h); q1.solve(); q1f = Plate(10, 1, shear="full", h=h); q1f.solve()
    say(f"         clamped  w_RM(Q2-SRI, N = 64) / w_K = {wc / wKc:.5f};  Q1 N = 10:  SRI {q1.center() / wKc:.4f} of w_K = {q1.center() / wc:.4f} of w_RM,  "
        f"full {q1f.center() / wKc:.4f} of w_K = {q1f.center() / wc:.4f} of w_RM")

> **✏️ Your turn (5 min).** How does $w_{RM}/w_K-1$ scale with $h$? Re-read the source's "SRI 1.184, naive 0.993" at $h=0.1$: which element is right? What is the lesson for choosing a
> reference?

> *your answer*

## 5 · Convergence rates against the exact solution  *(lecture §C2)*
Simply supported (hard), $h=0.01$, the error of the centre deflection against `navier_ss(D, F)` for Q1-SRI and Q2-SRI, $N$ = 4, 8, 16, 32.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from plate import *

E, nu, h = 210e3, 0.3, 0.01
D = E * h**3 / (12 * (1 - nu**2)); F = 5 / 6 * E / (2 * (1 + nu)) * h; wRM = navier_ss(D, F)
for deg in (1, 2):
    errs = []
    for N in (4, 8, 16, 32):
        p = Plate(N, deg, h=h, bc="ss_hard"); p.solve(); errs.append(abs(p.center() / wRM - 1))
    say(f"Q{deg}-SRI: relative errors " + " ".join(f"{e:.3e}" for e in errs) + ";  rates " + " ".join(f"{np.log2(errs[i] / errs[i + 1]):.2f}" for i in range(3)))

> **✏️ Your turn (3 min).** Which rates do you see, and are they what you expected for a point value of $w$? Would the rates survive $h=10^{-4}$?

> *your answer*

## 6 · LLM exercise: a clamped circular plate  *(lecture §D1–D2)*
Ask an LLM for a DOLFINx 0.11 script: clamped circular plate of radius $a=1$, $E=210\times10^3$, $\nu=0.3$, uniform load $q=1$, thicknesses $h$ = 0.1, 0.01, 0.001, a Gmsh mesh,
locking-free Reissner–Mindlin elements, the centre deflection compared with the **exact Reissner–Mindlin** value $w_0=\frac{qa^4}{64D}+\frac{qa^2}{4\kappa Gh}$. Paste it below, run it, judge
it. The reference cell follows.

In [ ]:
%%fenicsx -np 2
# Paste the LLM's script here (keep the %%fenicsx line). Then run the reference cell below.

In [ ]:
%%fenicsx -np 2
import numpy as np, ufl, basix.ufl, gmsh
from mpi4py import MPI
from dolfinx import fem, geometry
from dolfinx.io import gmsh as gio
from dolfinx.fem.petsc import LinearProblem
comm = MPI.COMM_WORLD
E, nu, a, q = 210e3, 0.3, 1.0, 1.0
gmsh.initialize(); gmsh.option.setNumber("General.Verbosity", 0)
if comm.rank == 0:                                                   # build on rank 0 only
    gmsh.model.occ.addDisk(0, 0, 0, a, a); gmsh.model.occ.synchronize()
    gmsh.model.addPhysicalGroup(2, [1], 1); gmsh.model.addPhysicalGroup(1, [1], 1)
    gmsh.option.setNumber("Mesh.CharacteristicLengthMax", 0.05)
    gmsh.option.setNumber("Mesh.RecombineAll", 1); gmsh.option.setNumber("Mesh.Algorithm", 8)   # quadrilaterals
    gmsh.model.mesh.generate(2); gmsh.model.mesh.setOrder(2)                                   # curved boundary
md = gio.model_to_mesh(gmsh.model, comm, 0, gdim=2); gmsh.finalize()
msh, ft = md.mesh, md.facet_tags
c = msh.basix_cell()
V = fem.functionspace(msh, basix.ufl.mixed_element([basix.ufl.element("Q", c, 2), basix.ufl.element("Q", c, 2, shape=(2,))]))
bcs = []
for i in (0, 1):
    Vc, _ = V.sub(i).collapse()
    bcs.append(fem.dirichletbc(fem.Function(Vc), fem.locate_dofs_topological((V.sub(i), Vc), 1, ft.find(1)), V.sub(i)))
p0 = np.array([[0.0, 0.0, 0.0]])
cc = geometry.compute_colliding_cells(msh, geometry.compute_collisions_points(geometry.bb_tree(msh, 2), p0), p0)
own = len(cc.links(0)) > 0 and cc.links(0)[0] < msh.topology.index_map(2).size_local
for h in (0.1, 0.01, 0.001):
    D = E * h**3 / (12 * (1 - nu**2)); F = 5 / 6 * E / (2 * (1 + nu)) * h
    (w, th), (v, ps) = ufl.TrialFunctions(V), ufl.TestFunctions(V)
    chi = lambda t: ufl.sym(ufl.grad(t)); M = lambda t: D * ((1 - nu) * chi(t) + nu * ufl.tr(chi(t)) * ufl.Identity(2))
    dxs = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": 2})
    A = ufl.inner(M(th), chi(ps)) * ufl.dx + F * ufl.dot(ufl.grad(w) - th, ufl.grad(v) - ps) * dxs
    uh = LinearProblem(A, q * v * ufl.dx, bcs=bcs, petsc_options_prefix=f"disk{int(1 / h)}_",
                       petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}).solve()
    w0 = comm.allreduce(float(uh.sub(0).eval(p0, [cc.links(0)[0]])[0]) if own else 0.0, op=MPI.SUM)
    ex = q * a**4 / (64 * D) + q * a**2 / (4 * F)
    if comm.rank == 0:
        print(f"h = {h}: {msh.topology.index_map(2).size_global} quadrilaterals;  w0 = {w0:.6e},  exact RM {ex:.6e},  ratio {w0 / ex:.6f};  "
              f"Kirchhoff part of the exact value {q * a**4 / (64 * D) / ex:.5f}")

> *your verdict on the LLM's code*